In [1]:
print("hi")

hi


In [1]:
import json
import re
import os
from typing import List
from PIL import Image
from pydantic import BaseModel
import pdfplumber
import torch
import cv2
import numpy as np
# from pipe_fn import pipe
from output_utils import save_split_output

from transformers import pipeline


from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)


# =========================
# CROP PT → REFERENCE
# =========================
def process_pdf_claims(pdf_path):

    claims = {}

    with pdfplumber.open(pdf_path) as pdf:

        for page_idx, page in enumerate(pdf.pages):

            start_hits = page.search("Patient:")
            end_hits   = page.search("CLAIM TOTALS")

            if not start_hits:
                start_hits = page.search("Claim")

            if not end_hits:
                end_hits = page.search("Claim Total:")

            if not start_hits or not end_hits:
                continue

            table_count = min(len(start_hits), len(end_hits))

            for i in range(table_count):

                start_y = start_hits[i]["top"] - 10
                end_y   = end_hits[i]["bottom"] + 20

                cropped = page.crop(
                    (
                        0,
                        start_y,
                        page.width,
                        end_y
                    )
                )

                img = cropped.to_image(
                    resolution=300
                ).original

                key = f"page_{page_idx+1}_claim_{i+1}"

                expected_rows = count_service_rows(
                    page,
                    start_y,
                    end_y
                )

                claims[key] = {
                    "image": img,
                    "expected_rows": expected_rows
                }

    return claims


def save_images(claims, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    paths = []

    for k, v in claims.items():
        path = os.path.join(out_dir, f"{k}.png")

        v["image"].save(path)

        paths.append({
            "image_path": path,
            "expected_rows": v["expected_rows"]
        })

    return paths

# =========================
# TABLE ENHANCEMENT
# =========================
def make_table(image_path):
    img = cv2.imread(image_path)
    gray = cv2.imread(image_path, 0)

    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

    sums = np.sum(thresh, axis=1)
    th = (thresh.shape[1] * 255) * 0.6
    lines = np.where(sums > th)[0]

    for l in lines:
        cv2.line(img, (0, l), (thresh.shape[1], l), (0, 0, 0), 1)

    return img

def convert_amounts_to_string(obj):

    amount_fields = {
        "billed_amount",
        "contractual_adjustments",
        "other_adjustments",
        "allowed",
        "patient_responsibility",
        "payment"
    }

    if isinstance(obj, dict):

        new_obj = {}

        for k, v in obj.items():

            if k in amount_fields:

                try:

                    clean_value = (
                        str(v)
                        .replace("$", "")
                        .replace(",", "")
                        .strip()
                    )

                    new_obj[k] = f"{float(clean_value):.2f}"

                except:
                    new_obj[k] = ""

            else:
                new_obj[k] = convert_amounts_to_string(v)

        return new_obj

    elif isinstance(obj, list):

        return [convert_amounts_to_string(i) for i in obj]

    return obj

def count_service_rows(
    page,
    region_top,
    region_bottom
):

    words = page.extract_words()

    service_rows = set()

    for w in words:

        text = w["text"].strip()

        if re.fullmatch(r"D\d{4}", text):

            y = round(
                float(w["top"]),
                1
            )

            if region_top <= y <= region_bottom:
                service_rows.add(y)

    return len(service_rows)



# =========================
# PROMPT
# =========================
def build_prompt(pdf_name):

    return """
You are extracting structured financial data from a dental EOB image.

STRICT RULES

OUTPUT ONLY VALID JSON.

DO NOT WRITE EXPLANATIONS.

DO NOT HALLUCINATE.

DO NOT CALCULATE VALUES.

DO NOT REMOVE DUPLICATE ROWS.

IF TWO ROWS LOOK IDENTICAL,
EXTRACT BOTH ROWS.

--------------------------------
COLUMN RULES
--------------------------------

provider

-> ONLY from the Service Provider: from the header.

date_of_service
→ ONLY from Service Date(s)

procedure_code
→ ONLY from Procedure (Modifiers)

billed_amount
→ ONLY from Billed Amount

contractual_adjustments
→ ONLY from Contractual Adjustments

other_adjustments
→ ONLY from Other Adjustments

allowed
→ ONLY from Allowed

patient_responsibility
→ ONLY from Patient Responsibility

payment
→ ONLY from Payment

--------------------------------
ROW RULES
--------------------------------

A SERVICE ROW EXISTS ONLY IF:

date_of_service exists
AND
procedure_code exists

If either is missing:
DO NOT create a service row.

--------------------------------
CLAIM TOTALS
--------------------------------

CLAIM TOTALS is NOT a service row.

Extract separately.

Do not calculate totals.

Extract only what appears in the CLAIM TOTALS row.
--------------------------------
OUTPUT
--------------------------------

{
  "patient_name": {
    "value": "",
    "confidence": 0.0
  },

  "provider": {
    "value": "",
    "confidence": 0.0
  },

  "date_of_service": {
    "value": "",
    "confidence": 0.0
  },

  "services": [
    {
      "procedure_code": {
        "value": "",
        "confidence": 0.0
      },

      "billed_amount": {
        "value": "",
        "confidence": 0.0
      },

      "contractual_adjustments": {
        "value": "",
        "confidence": 0.0
      },

      "other_adjustments": {
        "value": "",
        "confidence": 0.0
      },

      "allowed": {
        "value": "",
        "confidence": 0.0
      },

      "patient_responsibility": {
        "value": "",
        "confidence": 0.0
      },

      "payment": {
        "value": "",
        "confidence": 0.0
      }
    }
  ],

  "claim_totals": {
    "billed_amount": {
      "value": "",
      "confidence": 0.0
    },

    "contractual_adjustments": {
      "value": "",
      "confidence": 0.0
    },

    "other_adjustments": {
      "value": "",
      "confidence": 0.0
    },

    "allowed": {
      "value": "",
      "confidence": 0.0
    },

    "patient_responsibility": {
      "value": "",
      "confidence": 0.0
    },

    "payment": {
      "value": "",
      "confidence": 0.0
    }
  }
}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{
  "value": "",
  "confidence": ""
}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.
"""
def parse_amount(x):
    if x is None or x == "":
        return 0.0
    return float(
        str(x)
        .replace("$", "")
        .replace(",", "")
        .strip()
    )

def normalize_service_dates(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():

            # Handle date_of_service if it is a list
            if k == "date_of_service":
                if isinstance(v, list):
                    v = v[0] if v else ""

                if isinstance(v, str):
                    v = v.strip()

                    # Handle 03/10/26-03/10/26
                    if "-" in v:
                        left, right = v.split("-", 1)

                        if re.fullmatch(r"\d{2}/\d{2}/\d{2}", left) and \
                           re.fullmatch(r"\d{2}/\d{2}/\d{2}", right):
                            obj[k] = left

                        elif re.fullmatch(r"\d{2}/\d{2}", left) and \
                             re.fullmatch(r"\d{2}/\d{2}/\d{4}", right):
                            obj[k] = right
                        else:
                            obj[k] = v
                    else:
                        obj[k] = v

            # Handle service_date / service_dates
            elif k in ("service_date", "service_dates") and isinstance(v, str):
                v = v.strip()

                if "-" in v:
                    left, right = v.split("-", 1)

                    # 03/10-03/10/2026
                    if re.fullmatch(r"\d{2}/\d{2}", left) and \
                       re.fullmatch(r"\d{2}/\d{2}/\d{4}", right):
                        obj[k] = right

                    # 03/10/26-03/10/26
                    elif re.fullmatch(r"\d{2}/\d{2}/\d{2}", left) and \
                         re.fullmatch(r"\d{2}/\d{2}/\d{2}", right):
                        obj[k] = left
                    else:
                        obj[k] = v
                else:
                    obj[k] = v

            else:
                normalize_service_dates(v)

    elif isinstance(obj, list):
        for item in obj:
            normalize_service_dates(item)

    return obj

def compute_totals_from_services(services):
    return {
        "billed_amount": round(sum(parse_amount(s.get("billed_amount", "")) for s in services), 2),
        "contractual_adjustments": round(sum(parse_amount(s.get("contractual_adjustments", "")) for s in services), 2),
        "other_adjustments": round(sum(parse_amount(s.get("other_adjustments", "")) for s in services), 2),
        "allowed": round(sum(parse_amount(s.get("allowed", "")) for s in services), 2),
        "patient_responsibility": round(sum(parse_amount(s.get("patient_responsibility", "")) for s in services), 2),
        "payment": round(sum(parse_amount(s.get("payment", "")) for s in services), 2)
    }


def validate_patient_totals(patient: dict, patient_name: str, expected_row_count: int):

    services = patient.get("services", [])
    totals = patient.get("claim_totals",{})

    if not services:
        return False, "No services found", [{"error": "empty services"}]

    computed_totals = compute_totals_from_services(services)

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 80)

    # ===== FIELD VALIDATION =====
    for field, computed_value in computed_totals.items():

        extracted_value = round(parse_amount(totals.get(field, "")), 2)

        diff = round(computed_value - extracted_value, 2)
        match = abs(diff) <= 0.01   # 🔥 tolerance fix

        if match:
            icon = "✅"
            status = "MATCH"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True

            errors.append({
                "type": "field_mismatch",
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value,
                "difference": diff
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    # ===== ROW COUNT VALIDATION =====
    extracted_row_count = len(services)

    if expected_row_count == extracted_row_count:
        icon = "✅"
        status = "MATCH"
    else:
        icon = "❌"
        status = "MISMATCH"
        has_error = True

        errors.append({
            "type": "row_count_mismatch",
            "expected_rows": expected_row_count,
            "extracted_rows": extracted_row_count
        })

    line = f"{icon} {'total_record_rows':25s} computed={expected_row_count:<10} | extracted={extracted_row_count:<10} {status}"
    print(line)
    result_validation += "\n" + line

    print("-" * 80)

    # ===== FINAL STATUS =====
    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors
    else:
        print(f"✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, []

# =========================
# JSON CLEANER
# =========================
def extract_json(text):
    start = text.find("{")
    end = text.rfind("}") + 1
    return json.loads(text[start:end])

def save_json(data, output_path):
    with open(output_path, "w", encoding = "utf-8") as f:
        json.dump(data, f, indent = 2, ensure_ascii = False)

def check_claim_denied(pdf_path):

    denial_keywords = [
        "denied",
        "denial"
    ]

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(pdf.pages, start=1):

            full_text = page.extract_text()

            if not full_text:
                continue

            searchable_text = full_text.lower()

            for keyword in denial_keywords:

                if keyword in searchable_text:

                    print(
                        f"❌ Claim denied keyword found: "
                        f"'{keyword}' on page {page_num}"
                    )

                    return "denied"

    return "not denied"

def run_pipeline(pdf_path, output_dir="EOB_OUTPUT/Excellus_result", company_name = "Excellus"):

    pdf_name = os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    pdf_full_name = os.path.basename(pdf_path) 

    base_dir = os.path.join(output_dir, pdf_name)
    cropped_dir = os.path.join(base_dir, "cropped_images")

    os.makedirs(base_dir, exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    claims = process_pdf_claims(pdf_path)
    image_paths = save_images(claims, cropped_dir)

    final_prompt = build_prompt(pdf_name)
    is_denied = check_claim_denied(pdf_path)
    print(f"claim status :{is_denied}")

    results = []

    for idx, item in enumerate(image_paths):
        img_path = item["image_path"]
        expected_rows = item["expected_rows"]

        print(f"Processing {idx+1}/{len(image_paths)}")

        image = make_table(img_path)
        image = Image.fromarray(image).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": final_prompt}
                ]
            }
        ]

        with torch.no_grad():
            output = pipe(messages, max_new_tokens=1500, temperature= 0.0, do_sample=False)

        raw = output[0]["generated_text"]
        print("this is raw output", raw )

        if isinstance(raw, list):
            raw = raw[-1]["content"]


        try:
            parsed = extract_json(raw)

            # ADD — unwrap value/confidence wrappers before amount/date cleanup touches the fields
            model_confidence = calculate_model_confidence(parsed)
            parsed = _unwrap_vlm_output(parsed)
            parsed["_model_confidence"] = model_confidence

            parsed = convert_amounts_to_string(parsed)
            parsed = normalize_service_dates(parsed)
            results.append(parsed)
            parsed["_expected_rows"] = expected_rows
            print("✔ extracted")
        except Exception as e:
            print(f"❌ json failed:{e}")

        for patient in results:

            is_valid, log, errors = validate_patient_totals(
            patient=patient,
            patient_name=patient.get("patient_name", ""),
            expected_row_count=patient.get("_expected_rows", 0)  # or your external expected count
        )

            patient["validation"] = {
                "status": is_valid,
                "errors": errors
            }
    confidence_score = calculate_eob_confidence(results)   # ADD — before popping temp keys

    for patient in results:
        patient.pop("_expected_rows", None)
        patient.pop("_model_confidence", None)   # ADD

    final =[
        {
        "eob_id": pdf_name,
        "file_name": pdf_full_name,          # ADD
        "claim_status": is_denied,
        "payor": "Excellus",
        "confidence_score": confidence_score,  # ADD
        "patients": results,
        }
    ]

    success_path, failed_path = save_split_output(
                        final,
                        company_name=company_name,
                        pdf_name=pdf_name,
                        pdf_path=pdf_path,
                        cropped_dir=cropped_dir,
                    )
                
    print(f"\n📁 Cropped images : {cropped_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final

W0901 19:02:14.512000 3494215 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 19:02:14.527000 3494215 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Excellus/pdf/Pmt_EOP_296687623.pdf")

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


claim status :not denied
Processing 1/1


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


this is raw output [{'role': 'user', 'content': [{'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=2550x614 at 0x79A438AE8770>}, {'type': 'text', 'text': '\nYou are extracting structured financial data from a dental EOB image.\n\nSTRICT RULES\n\nOUTPUT ONLY VALID JSON.\n\nDO NOT WRITE EXPLANATIONS.\n\nDO NOT HALLUCINATE.\n\nDO NOT CALCULATE VALUES.\n\nDO NOT REMOVE DUPLICATE ROWS.\n\nIF TWO ROWS LOOK IDENTICAL,\nEXTRACT BOTH ROWS.\n\n--------------------------------\nCOLUMN RULES\n--------------------------------\n\nprovider\n\n-> ONLY from the Service Provider: from the header.\n\ndate_of_service\n→ ONLY from Service Date(s)\n\nprocedure_code\n→ ONLY from Procedure (Modifiers)\n\nbilled_amount\n→ ONLY from Billed Amount\n\ncontractual_adjustments\n→ ONLY from Contractual Adjustments\n\nother_adjustments\n→ ONLY from Other Adjustments\n\nallowed\n→ ONLY from Allowed\n\npatient_responsibility\n→ ONLY from Patient Responsibility\n\npayment\n→ ONLY from Payment\n\n----------

[{'eob_id': '296687623',
  'file_name': 'Pmt_EOP_296687623.pdf',
  'claim_status': 'not denied',
  'payor': 'Excellus',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'LESLEE MABEE',
    'provider': 'Duc Tang',
    'date_of_service': '11/10/22',
    'services': [{'procedure_code': 'D0120',
      'billed_amount': '65.63',
      'contractual_adjustments': '0.00',
      'other_adjustments': '0.00',
      'allowed': '34.11',
      'patient_responsibility': '31.52',
      'payment': '34.11'},
     {'procedure_code': 'D1110',
      'billed_amount': '117.39',
      'contractual_adjustments': '0.00',
      'other_adjustments': '0.00',
      'allowed': '75.40',
      'patient_responsibility': '41.99',
      'payment': '75.40'},
     {'procedure_code': 'D0330',
      'billed_amount': '141.85',
      'contractual_adjustments': '0.00',
      'other_adjustments': '0.00',
      'allowed': '79.10',
      'patient_responsibility': '62.75',
      'payment': '79.10'}],
    'claim_totals': 

In [3]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Excellus/pdf/Pmt_EOP_296687623.pdf")

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


claim status :not denied
Processing 1/1


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


this is raw output [{'role': 'user', 'content': [{'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=2550x614 at 0x731EA42E2DB0>}, {'type': 'text', 'text': '\nYou are extracting structured financial data from a dental EOB image.\n\nSTRICT RULES\n\nOUTPUT ONLY VALID JSON.\n\nDO NOT WRITE EXPLANATIONS.\n\nDO NOT HALLUCINATE.\n\nDO NOT CALCULATE VALUES.\n\nDO NOT REMOVE DUPLICATE ROWS.\n\nIF TWO ROWS LOOK IDENTICAL,\nEXTRACT BOTH ROWS.\n\n--------------------------------\nCOLUMN RULES\n--------------------------------\n\nprovider\n\n-> ONLY from the Service Provider: from the header.\n\ndate_of_service\n→ ONLY from Service Date(s)\n\nprocedure_code\n→ ONLY from Procedure (Modifiers)\n\nbilled_amount\n→ ONLY from Billed Amount\n\ncontractual_adjustments\n→ ONLY from Contractual Adjustments\n\nother_adjustments\n→ ONLY from Other Adjustments\n\nallowed\n→ ONLY from Allowed\n\npatient_responsibility\n→ ONLY from Patient Responsibility\n\npayment\n→ ONLY from Payment\n\n----------

[{'eob_id': '296687623',
  'file_name': 'Pmt_EOP_296687623.pdf',
  'claim_status': 'not denied',
  'payor': 'Excellus',
  'confidence_score': 99.0,
  'patients': [{'patient_name': 'LESLEE MABEE',
    'provider': 'Duc Tang',
    'date_of_service': '11/10/22',
    'services': [{'procedure_code': 'D0120',
      'billed_amount': '65.63',
      'contractual_adjustments': '0.00',
      'other_adjustments': '0.00',
      'allowed': '34.11',
      'patient_responsibility': '31.52',
      'payment': '34.11'},
     {'procedure_code': 'D1110',
      'billed_amount': '117.39',
      'contractual_adjustments': '0.00',
      'other_adjustments': '0.00',
      'allowed': '75.40',
      'patient_responsibility': '41.99',
      'payment': '75.40'},
     {'procedure_code': 'D0330',
      'billed_amount': '141.85',
      'contractual_adjustments': '0.00',
      'other_adjustments': '0.00',
      'allowed': '79.10',
      'patient_responsibility': '62.75',
      'payment': '79.10'}],
    'claim_totals': {